# 03 — Profiling & Silver Cleaning.


O Data Profiling vai ajudar-nos a entender a qualidade dos dados (nulos, duplicados e distribuição). A Silver Cleaning aplicará as regras de negócio para corrigir esses problemas e ajustar os esquemas (schema casting).


### 3.1 Setup inicial
Nesta fase vamos realizar os imports, carregar as tabelas previamente guardadas em formato Delta e definir os caminhos base de cada uma delas.


In [0]:
# Setup inicial
# imports necessários, definição de caminhos base necessários e lista com as tabelas a trabalhar
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Caminhos Delta necessários para correr este notebook de forma independente
bronze_delta_path = "/Volumes/main/default/faers_data/delta/bronze"
silver_delta_path = "/Volumes/main/default/faers_data/delta/silver"

tables = ["demo", "drug", "reac", "outc"]


In [0]:
# Primeiro carregamos as tabelas Delta da camada Bronze
bronze_dfs = {}
for table in tables:
    bronze_dfs[table] = spark.read.format("delta").load(f"{bronze_delta_path}/{table}/")


In [0]:
# visualização das tabelas
for table in tables:
    print(f"\nTabela: {table}")
    display(bronze_dfs[table].limit(5))


### 3.2 Schema Casting (Conversão de Tipos de Dados)

Na camada Bronze, todas as colunas foram ingeridas temporariamente como `string` para garantir a fidelidade aos ficheiros originais. Agora, na camada Silver, é necessário atribuir os tipos de dados semânticos corretos (Datas, Números Inteiros e Decimais).

**Principais Transformações:**
1. **Datas:** Os ficheiros FAERS utilizam o formato `AAAAMMDD`. Colunas como `event_dt` ou `fda_dt` serão convertidas para `DateType`.*
2. **Métricas Clínicas e Doses:** Colunas quantitativas como `age` (idade), `wt` (peso) e `dose_amt` (quantidade da dose) serão convertidas para `IntergerType` e `DoubleType` respectivamente, para permitir agregações matemáticas (médias, distribuições) na camada Gold.
3. As variáveis categóricas e identificadores (como `primaryid`, `pt`, `outc_cod`) mantêm-se como `string`.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType

silver_dfs = {}
print("=== SCHEMA CASTING E GRAVAÇÃO SILVER ===\n")

for table_name, df in bronze_dfs.items():
    df_cast = df
    
    # Transformações específicas para a tabela DEMO
    if table_name == "demo":
        df_cast = df_cast.withColumn("event_dt", F.to_date(F.col("event_dt"), "yyyyMMdd")) \
                         .withColumn("mfr_dt", F.to_date(F.col("mfr_dt"), "yyyyMMdd")) \
                         .withColumn("init_fda_dt", F.to_date(F.col("init_fda_dt"), "yyyyMMdd")) \
                         .withColumn("fda_dt", F.to_date(F.col("fda_dt"), "yyyyMMdd")) \
                         .withColumn("rept_dt", F.to_date(F.col("rept_dt"), "yyyyMMdd")) \
                         .withColumn("age", F.col("age").cast(IntegerType())) \
                         .withColumn("wt", F.col("wt").cast(DoubleType()))
                         
    # Transformações específicas para a tabela DRUG
    elif table_name == "drug":
        df_cast = df_cast.withColumn("exp_dt", F.to_date(F.col("exp_dt"), "yyyyMMdd")) \
                         .withColumn("dose_amt", F.col("dose_amt").cast(DoubleType())) \
                         .withColumn("cum_dose_chr", F.col("cum_dose_chr").cast(DoubleType()))
                         
    # As tabelas REAC e OUTC contêm apenas identificadores e códigos em texto, 
    # pelo que não necessitam de casting numérico/temporal.
    
    silver_dfs[table_name] = df_cast
    print(f"Schema atualizado para a tabela: {table_name.upper()}")

print("\n--- A Iniciar Gravação na Camada Silver ---")

# Gravação em formato Delta
for table_name, df in silver_dfs.items():
    output_path = f"{silver_delta_path}/{table_name}"
    
    (
        df.write
          .mode("overwrite")
          .format("delta")
          .option("overwriteSchema", "true")
          .save(output_path)
    )
    print(f"✅ Tabela {table_name.upper()} gravada com sucesso em: {output_path}")





### 3.3 Profiling inicial
Após verificarmos que as tabelas bronze foram corretamente carregadas vamos começar a fazer um profiling inicial.

Nesta fase queremos perceber a estrutura e qualidade dos dados antes de definir regras de limpeza.


### 3.3.1 Contagens e schemas

In [0]:
# Faz-se uma contagem do nº de registos e de colunas de cada tabela na fase bronze.
# Neste caso, todas as tabelas foram carregadas com um schema que define todas as colunas como string.
# Ainda assim, é importante realizar uma última verificação.
for table, df in silver_dfs.items():
    print(f"Tabela {table.upper()} apresenta {bronze_dfs[table].count():,} registos.")
    print(f"Tabela {table.upper()} apresenta {len(bronze_dfs[table].columns)} colunas.")
    print(f"\nSchema da tabela {table.upper()}:")
    df.printSchema()


### 3.2.2 Verificação de nulos
Após a leitura das tabelas e a análise dos respetivos schemas, procede-se à verificação de valores nulos em cada coluna.

O objetivo desta etapa é avaliar a completude dos dados antes da aplicação das regras de limpeza da camada Silver. Para cada tabela, será calculado o número de valores nulos por coluna e a respetiva percentagem face ao total de registos.

Esta análise permite distinguir entre nulos em campos críticos, como `primaryid` e `caseid`, que podem comprometer a integridade relacional dos dados, e nulos em campos opcionais ou clinicamente informativos, cuja ausência pode ser esperada e deve ser preservada ou tratada com cautela.


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

for table_name, df in silver_dfs.items():
    print(f"--- Análise Detalhada de Nulos: {table_name.upper()} ---")
    
    total_rows = df.count()
    
    if total_rows == 0:
        print("A tabela está vazia.\n")
        continue

    # 1. Calcula a contagem de nulos para todas as colunas
    null_exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    null_counts_row = df.select(*null_exprs).collect()[0]
    
    # 2. Constrói a lista APENAS com colunas que têm nulos
    summary_data = []
    for c in df.columns:
        n_nulls = null_counts_row[c]
        
        # Só adiciona à lista se tiver pelo menos 1 nulo
        if n_nulls > 0:
            pct_nulls = round((n_nulls / total_rows) * 100, 2)
            summary_data.append((c, n_nulls, pct_nulls))
            
    # 3. Se a lista estiver vazia (zero nulos na tabela inteira), avisa e avança para a próxima tabela
    if not summary_data:
        print("🎉 Não existem colunas com valores nulos nesta tabela!\n")
        continue
        
    # 4. Define o esquema para o novo DataFrame
    schema = StructType([
        StructField("Nome_Coluna", StringType(), True),
        StructField("Qtd_Nulos", LongType(), True),
        StructField("%_Nulos", DoubleType(), True)
    ])
    
    # 5. Cria o DataFrame, ordena de forma decrescente e exibe
    summary_df = spark.createDataFrame(summary_data, schema)
    summary_df = summary_df.orderBy(F.col("Qtd_Nulos").desc())
    
    display(summary_df)


### 3.2.3 Verificação de duplicados

A análise de duplicados será usada para definir a estratégia de limpeza na camada Silver.

Na camada Silver serão removidos duplicados exatos e serão mantidos os identificadores necessários para preservar relações entre tabelas. *A deduplicação por chave lógica será aplicada com cuidado, uma vez que algumas tabelas FAERS podem conter múltiplos registos válidos por caso, medicamento, reação ou desfecho.*


In [0]:
for table, df in bronze_dfs.items():
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(f"{table.upper()}")
    print(f"Total: {total_rows}")
    print(f"Distintos: {distinct_rows}")
    print(f"Duplicados exatos: {duplicate_rows}")
    print("-" * 40)


### 3.2.3 Duplicados por chave lógica

Ao contrário dos duplicados exatos, onde todas as colunas da linha são iguais, os duplicados por chave lógica ocorrem quando existem várias linhas com a mesma combinação de campos identificadores. Estes casos podem representar duplicação real, mas também podem refletir a estrutura natural dos dados FAERS.

Nesta análise, não serão removidos registos automaticamente. O objetivo é apenas identificar situações em que a mesma chave lógica aparece mais do que uma vez, para compreender se esses casos exigem tratamento posterior na camada Silver.

As chaves lógicas consideradas são:

| Tabela | Chave lógica utilizada | Interpretação |
|---|---|---|
| `demo` | `primaryid`, `caseid`, `caseversion` | identifica uma versão específica de um caso |
| `drug` | `primaryid`, `caseid`, `drug_seq` | identifica um medicamento dentro de um caso |
| `reac` | `primaryid`, `caseid`, `pt` | identifica uma reação reportada num caso |
| `outc` | `primaryid`, `caseid`, `outc_cod` | identifica um outcome associado a um caso |

Esta análise permite perceber se existem combinações de chaves repetidas e avaliar se devem ser mantidas, investigadas ou removidas numa fase posterior. Em tabelas como `drug`, `reac` e `outc`, a existência de várias linhas por caso pode ser esperada, uma vez que um mesmo caso pode estar associado a múltiplos medicamentos, reações ou outcomes.

In [0]:
# definir função para identificar duplicados por chave lógica
def analisar_duplicados_chave_logica(df, table_name, key_cols, show_results=True):
    """
    Analisa duplicados por chave lógica numa tabela.

    Parâmetros:
    df : DataFrame
        DataFrame a analisar.
    table_name : str
        Nome da tabela, usado apenas para identificação no output.
    key_cols : list
        Lista de colunas que compõem a chave lógica.
    show_results : bool
        Se True, mostra o resultado com display().

    Retorna:
    DataFrame com as combinações de chave lógica que aparecem mais do que uma vez.
    """

    print(f"--- Duplicados por chave lógica: {table_name.upper()} ---")

    # Confirma se todas as colunas da chave existem na tabela
    missing_cols = [c for c in key_cols if c not in df.columns]

    if missing_cols:
        print(f"Colunas em falta para esta análise: {missing_cols}\n")
        return None

    # Agrupa pela chave lógica e identifica combinações repetidas
    duplicate_keys_df = (
        df.groupBy(key_cols)
          .count()
          .filter(F.col("count") > 1)
          .orderBy(F.desc("count"))
    )

    total_duplicate_keys = duplicate_keys_df.count()

    print(f"Número de chaves lógicas com mais de um registo: {total_duplicate_keys}")

    if show_results:
        display(duplicate_keys_df.limit(5))

    return duplicate_keys_df

In [0]:
# Define um dicionário com as chaves lógicas a usar em cada tabela.
logical_keys = {
    "demo": ["primaryid", "caseid", "caseversion"],
    "drug": ["primaryid", "caseid", "drug_seq"],
    "reac": ["primaryid", "caseid", "pt"],
    "outc": ["primaryid", "caseid", "outc_cod"]
}

duplicate_keys_dfs = {}

# analise duplicados por chave lógica em cada tabela.
for table_name, key_cols in logical_keys.items():
    df = silver_dfs[table_name]

    duplicate_keys_dfs[table_name] = analisar_duplicados_chave_logica(
        df=df,
        table_name=table_name,
        key_cols=key_cols,
        show_results=True
    )

### 3.2.4 Valores distintos em variáveis categóricas

Nesta etapa serão analisados os valores distintos existentes em algumas colunas categóricas relevantes das tabelas FAERS.

O objetivo é perceber que códigos e categorias aparecem nos dados antes da aplicação das regras de limpeza da camada Silver. Esta análise permite identificar inconsistências como diferenças de capitalização, espaços em branco, valores pouco frequentes, categorias desconhecidas ou códigos que possam necessitar de normalização.

Serão analisadas sobretudo colunas com significado categórico, como sexo, país do reporter, tipo de reporter, papel do medicamento no caso, via de administração, resultados clínicos e códigos de desfecho.

Para cada coluna selecionada, será apresentada a contagem de ocorrências por valor distinto, ordenada de forma decrescente. Esta informação será usada posteriormente para justificar transformações como `trim`, `upper` e o preenchimento de valores desconhecidos com códigos como `UNK` ou `U`.

In [0]:
import pyspark.sql.functions as F

categorical_cols = {
    "demo": ["sex", "occp_cod", "reporter_country", "e_sub"],
    "drug": ["role_cod", "route", "dechal", "rechal", "dose_freq"],
    "reac": ["pt"],
    "outc": ["outc_cod"]
}

for table_name, cols in categorical_cols.items():
    print(f"--- Valores distintos em variáveis categóricas: {table_name.upper()} ---")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            print(f"A coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"Coluna: {col_name}")
        
        distinct_values_df = (
            df.groupBy(col_name)
              .count()
              .orderBy(F.desc("count"))
              .limit(30)
        )
        
        display(distinct_values_df)

### 3.3.5 Análise de Variáveis Numéricas

Após a análise de variáveis categóricas, é importante explorar as variáveis quantitativas para identificar:
- **Distribuições**: valores mínimos, máximos, médias e medianas.
- **Outliers**: valores extremos ou clinicamente implausíveis (ex: idades negativas, pesos acima de 500 kg).
- **Missing patterns**: verificar se existem padrões de ausência em doses ou métricas clínicas.

As principais variáveis numéricas nas tabelas FAERS são:

| Tabela | Variável | Descrição |
|--------|----------|------------|
| `demo` | `age` | Idade do paciente |
| `demo` | `wt` | Peso do paciente (kg) |
| `drug` | `dose_amt` | Quantidade da dose administrada |
| `drug` | `cum_dose_chr` | Dose cumulativa |

Esta análise ajudará a definir regras de limpeza para valores extremos e a decidir estratégias de imputação ou filtragem na camada Silver.

In [0]:
import pyspark.sql.functions as F

print("=== DISTRIBUIÇÃO DE CÓDIGOS DE UNIDADE ===\n")

# Analisar distribuição de age_cod na tabela demo
print("--- Tabela: DEMO | Coluna: age_cod ---")
df_demo = silver_dfs["demo"]

age_cod_dist = (
    df_demo.groupBy("age_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(age_cod_dist)

print("\nSignificado dos códigos de age_cod:")
print("  YR  = Anos (Years)")
print("  DY  = Dias (Days)")
print("  DEC = Décadas (Decades)")
print("  MON = Meses (Months)")
print("  WK  = Semanas (Weeks)")
print("  HR  = Horas (Hours)")

print("\n" + "-"*60 + "\n")

# Analisar distribuição de wt_cod na tabela demo
print("--- Tabela: DEMO | Coluna: wt_cod ---")

wt_cod_dist = (
    df_demo.groupBy("wt_cod")
    .agg(
        F.count("*").alias("count"),
        F.round((F.count("*") / df_demo.count()) * 100, 2).alias("percentage")
    )
    .orderBy(F.desc("count"))
)

display(wt_cod_dist)

print("\nSignificado dos códigos de wt_cod:")
print("  KG  = Quilogramas (Kilograms)")
print("  LBS = Libras (Pounds)")
print("  GMS = Gramas (Grams)")

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType

# Definir colunas numéricas por tabela
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n{'='*60}")
    print(f"Análise de Variáveis Numéricas: {table_name.upper()}")
    print(f"{'='*60}\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            print(f"⚠️  Coluna '{col_name}' não existe na tabela {table_name}.\n")
            continue
        
        print(f"\n--- Coluna: {col_name.upper()} ---")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Estatísticas descritivas usando describe()
        stats_df = df_filtered.select(col_name).describe()
        display(stats_df)
        
        # Análise adicional: valores negativos e zeros
        total_rows = df.count()
        total_rows_filtered = df_filtered.count()
        negative_count = df_filtered.filter(F.col(col_name) < 0).count()
        zero_count = df_filtered.filter(F.col(col_name) == 0).count()
        null_count = df_filtered.filter(F.col(col_name).isNull()).count()
        
        print(f"\n📊 Análise de Qualidade:")
        if col_name in ["age", "wt"]:
            print(f"   Total de registos (tabela completa): {total_rows:,}")
            print(f"   Total de registos após filtro: {total_rows_filtered:,}")
        else:
            print(f"   Total de registos: {total_rows:,}")
        print(f"   Valores nulos: {null_count:,} ({round(null_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores negativos: {negative_count:,} ({round(negative_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores zero: {zero_count:,} ({round(zero_count/total_rows_filtered*100, 2) if total_rows_filtered > 0 else 0}%)")
        print(f"   Valores válidos (não-nulos e positivos): {total_rows_filtered - null_count - negative_count:,}")
        print("-" * 60)

In [0]:
import pyspark.sql.functions as F

print("\n" + "="*80)
print("ANÁLISE DE OUTLIERS - PERCENTIS E VALORES EXTREMOS")
print("="*80 + "\n")

# Analisar percentis para detectar outliers
numerical_cols = {
    "demo": ["age", "wt"],
    "drug": ["dose_amt", "cum_dose_chr"]
}

for table_name, cols in numerical_cols.items():
    print(f"\n--- Tabela: {table_name.upper()} ---\n")
    
    df = silver_dfs[table_name]
    
    for col_name in cols:
        if col_name not in df.columns:
            continue
        
        print(f"Coluna: {col_name.upper()}")
        
        # Para a coluna 'age', filtrar apenas registos onde age_cod = 'YR'
        # Para a coluna 'wt', filtrar apenas registos onde wt_cod = 'KG'
        if col_name == "age" and "age_cod" in df.columns:
            print("⚠️  Filtro aplicado: age_cod = 'YR' (apenas idades em anos)\n")
            df_filtered = df.filter(F.col("age_cod") == "YR")
        elif col_name == "wt" and "wt_cod" in df.columns:
            print("⚠️  Filtro aplicado: wt_cod = 'KG' (apenas pesos em quilogramas)\n")
            df_filtered = df.filter(F.col("wt_cod") == "KG")
        else:
            df_filtered = df
        
        # Calcular percentis (1%, 5%, 25%, 50%, 75%, 95%, 99%)
        percentiles = df_filtered.stat.approxQuantile(
            col_name, 
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99], 
            0.01
        )
        
        print(f"   P1:  {percentiles[0]}")
        print(f"   P5:  {percentiles[1]}")
        print(f"   P25: {percentiles[2]}")
        print(f"   P50 (Mediana): {percentiles[3]}")
        print(f"   P75: {percentiles[4]}")
        print(f"   P95: {percentiles[5]}")
        print(f"   P99: {percentiles[6]}")
        
        # Mostrar os 10 valores mais extremos (máximos)
        print(f"\n   Top 10 valores mais altos:")
        top_values = (
            df_filtered.select(col_name)
              .filter(F.col(col_name).isNotNull())
              .orderBy(F.desc(col_name))
              .limit(10)
        )
        display(top_values)
        
        print("-" * 60)

### 3.4.1 Tratamento de Duplicados

O profiling revelou a presença de duplicados exatos, com particular incidência na tabela `REAC` (mais de 100 mil registos). No contexto do FAERS, isto ocorre frequentemente devido a redundâncias no preenchimento do formulário original ou em submissões de acompanhamento (*follow-ups*) onde os mesmos sintomas são recarregados.

**Estratégia de Limpeza:**
Uma vez que são duplicados exatos (todas as colunas contêm os mesmos valores), estes registos não acrescentam qualquer contexto clínico novo. Pelo contrário, mantê-los causaria enviesamento e dupla contagem (*double-counting*) na fase de modelação ou na criação de dashboards. 

Aplica-se a função `dropDuplicates()` a todas as tabelas para garantir a integridade da camada Silver, mantendo apenas registos únicos para cada combinação de caso, medicamento e reação.


In [0]:
print("=== Tratamento De Duplicados Exatos===\n")

# Dicionário temporário para guardar os DataFrames sem duplicados
dedup_dfs = {}

for table_name, df in silver_dfs.items():
    # 1. Contagem inicial (antes da remoção)
    total_rows_antes = df.count()
    
    # 2. Remover duplicados exatos (avalia todas as colunas por defeito)
    df_dedup = df.dropDuplicates()
    
    # 3. Contagem final e cálculo da diferença
    total_rows_depois = df_dedup.count()
    duplicados_removidos = total_rows_antes - total_rows_depois
    
    # 4. Guardar o DataFrame limpo no novo dicionário
    dedup_dfs[table_name] = df_dedup
    
    # Mostrar resultados
    print(f"--- Tabela: {table_name.upper()} ---")
    print(f"Total antes: {total_rows_antes:,}")
    print(f"Duplicados removidos: {duplicados_removidos:,}")
    print(f"Total depois: {total_rows_depois:,}\n")

# Atualizar o dicionário principal com os dados agora sem nulos críticos e sem duplicados
silver_dfs = dedup_dfs


### 3.6 Validação Final da Camada Silver (Data Quality Check)

Antes de darmos a camada Silver como concluída, realizamos uma auditoria final diretamente nos ficheiros Delta que foram gravados no Unity Catalog/Volume. 

Esta validação garante que:
1. O motor Spark consegue ler as tabelas gravadas sem corrupção.
2. O **Schema Casting** foi persistido corretamente (verificando os tipos `date` e `double`).
3. Visualizamos uma amostra real dos dados já limpos de nulos críticos, sem duplicados e com a tipagem correta, prontos para alimentar a camada Gold.


In [0]:
print("=== AUDITORIA E VALIDAÇÃO DA CAMADA SILVER (DELTA) ===\n")

for table in tables:
    silver_path = f"{silver_delta_path}/{table}"
    
    print(f" Matriz de Validação para a tabela: {table.upper()}")
    
    # Ler diretamente do caminho Delta gravado
    df_silver = spark.read.format("delta").load(silver_path)
    
    # 1. Contagem total de linhas salvas
    total_rows = df_silver.count()
    print(f"   -> Total de registos persistidos: {total_rows:,}")
    print(f"   -> Total de colunas: {len(df_silver.columns)}")
    
    # 2. Print do Schema para validar visualmente o Casting
    print("   -> Estrutura do Schema:")
    df_silver.printSchema()
    
    # 3. Mostrar uma amostra rápida dos dados limpos
    print(f"   -> Amostra dos primeiros 3 registos de {table.upper()}:")
    display(df_silver.limit(3))
    
    print("-" * 80)
